In [23]:
!pip install joblib datasets

In [25]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset("civil_comments")

df = pd.DataFrame(dataset["train"])

print(df.columns)

Index(['text', 'toxicity', 'severe_toxicity', 'obscene', 'threat', 'insult',
       'identity_attack', 'sexual_explicit'],
      dtype='object')


In [26]:
dataset = load_dataset("civil_comments")

df = pd.DataFrame(dataset["train"])

df = df[
    ["text", "toxicity", "severe_toxicity", "obscene",
     "threat", "insult", "identity_attack", "sexual_explicit"]
]

df = df.dropna()

df.head()

,text,toxicity,severe_toxicity,obscene,threat,insult,identity_attack,sexual_explicit
0,"This is so cool. It's like, 'would you want yo...",0.000000,0.000000,0.0,0.0,0.00000,0.000000,0.0
1,Thank you!! This would make my life a lot less...,0.000000,0.000000,0.0,0.0,0.00000,0.000000,0.0
2,This is such an urgent design problem; kudos t...,0.000000,0.000000,0.0,0.0,0.00000,0.000000,0.0
3,Is this something I'll be able to install on m...,0.000000,0.000000,0.0,0.0,0.00000,0.000000,0.0
4,haha you guys are a bunch of losers.,0.893617,0.021277,0.0,0.0,0.87234,0.021277,0.0


In [27]:
df["toxicity_label"] = (df["toxicity"] > 0.5).astype(int)

df["bias_label"] = (
    (df["identity_attack"] > 0.5) |
    (df["insult"] > 0.7)
).astype(int)

df = df[["text", "toxicity_label", "bias_label"]]

df.head()

,text,toxicity_label,bias_label
0,"This is so cool. It's like, 'would you want yo...",0,0
1,Thank you!! This would make my life a lot less...,0,0
2,This is such an urgent design problem; kudos t...,0,0
3,Is this something I'll be able to install on m...,0,0
4,haha you guys are a bunch of losers.,1,1


In [28]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words="english"
)

X = vectorizer.fit_transform(df["text"])

In [29]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

models = {}

for target in ["toxicity_label", "bias_label"]:
    y = df[target]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    model = LogisticRegression(max_iter=200)
    model.fit(X_train, y_train)

    models[target] = model

    print(target, "trained")

toxicity_label trained ✔
bias_label trained ✔


In [30]:
def predict(text):
    vec = vectorizer.transform([text])

    return {
        "toxicity": int(models["toxicity_label"].predict(vec)[0]),
        "bias_signal": int(models["bias_label"].predict(vec)[0])
    }

In [31]:
def explain(text, model):
    vec = vectorizer.transform([text])

    feature_names = vectorizer.get_feature_names_out()

    indices = vec.nonzero()[1]
    values = vec.data

    coefs = model.coef_[0]

    word_scores = []

    for idx, tfidf_val in zip(indices, values):
        word = feature_names[idx]
        score = tfidf_val * coefs[idx]
        word_scores.append((word, score))

    word_scores.sort(key=lambda x: abs(x[1]), reverse=True)

    return word_scores[:10]

In [33]:
def predict_with_confidence(text):
    vec = vectorizer.transform([text])

    results = {}

    for name, model in models.items():
        proba = model.predict_proba(vec)[0][1]
        results[name] = float(proba)

    return results

In [35]:
sample = "The government should support women in education"

print("Prediction:")
print(predict(sample))

print("\nTop contributing words:")
print(explain(sample, models["toxicity_label"]))

Prediction:
{'toxicity': 0, 'bias_signal': 0}

Top contributing words:
[('women', np.float64(0.9586967616631002)), ('government', np.float64(-0.2644353074821233)), ('support', np.float64(-0.1275292207642349)), ('education', np.float64(-0.06464190976415637))]


In [36]:
print("""
Insight:
This model learns statistical correlations from data.
It does NOT understand bias or toxicity like humans.
Words like 'women' may appear as signals due to dataset imbalance.
""")


Insight:
This model learns statistical correlations from data.
It does NOT understand bias or toxicity like humans.
Words like 'women' may appear as signals due to dataset imbalance.



In [34]:
print("These words influenced model decision (not real-world bias):")

⚠ These words influenced model decision (not real-world bias):
